In [1]:
#Import your libraries here

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import itertools
import os
import sys
import scipy.optimize as sco

In [2]:
current_dir = os.path.abspath('')
project_root = os.path.abspath(os.path.join(current_dir, '../'))

if project_root not in sys.path:
    sys.path.append(project_root)

os.chdir(project_root)

print(f"Working directory set to: {os.getcwd()}")

Working directory set to: D:\users\kamen.dimitrov\desktop\softuni\AI_and_ML_upskill_program\Data Science\08_final_project


In [3]:
import importlib
from src.data_pipeline_utils import data_fetching_handling as data_pipe
import src.plotting_utils.plotting_utils as plot_utils
importlib.reload(plot_utils)
importlib.reload(data_pipe)

<module 'src.data_pipeline_utils.data_fetching_handling' from 'D:\\users\\kamen.dimitrov\\desktop\\softuni\\AI_and_ML_upskill_program\\Data Science\\08_final_project\\src\\data_pipeline_utils\\data_fetching_handling.py'>

In [4]:
tickers = [
    "AAPL", "GOOG", "MSFT", "NVDA",
    "JPM", "BAC", "F", "UPS", "WMT", "TGT",
    "VZ", "T", "FE", "PFE", "JNJ", "DIS",
    "V", "MCD", "NKE", "XOM", "CVX",
    "CAT", "DE", "LMT", "AMD", "INTC", "ORCL", 
    "CRM", "CB", "PG"
]
returns_df =  data_pipe.build_returns_df(tickers)
returns_df

,AAPL,GOOG,MSFT,NVDA,JPM,BAC,F,UPS,WMT,TGT,...,CVX,CAT,DE,LMT,AMD,INTC,ORCL,CRM,CB,PG
Date,,,,,,,,,,,,,,,,,,,,,
2016-04-01,0.009133,0.006636,0.006137,0.014489,0.010916,0.002954,-0.030077,-0.004943,0.008288,0.005817,...,-0.012022,0.003261,-0.006385,0.007242,-0.007042,0.003086,0.006092,0.025409,0.014167,0.014714
2016-04-04,0.010221,-0.006180,-0.002523,-0.009729,-0.003877,-0.003694,-0.023167,-0.000668,0.000579,-0.005573,...,-0.008844,-0.014032,-0.007874,0.004561,0.000000,-0.013965,-0.002189,-0.000528,-0.002154,-0.003838
2016-04-05,-0.011859,-0.010101,-0.015820,-0.001397,-0.014291,-0.023971,-0.002347,-0.009195,-0.006680,-0.002433,...,-0.008167,-0.006360,0.002237,0.006626,-0.025046,-0.003130,-0.013235,-0.014774,-0.022136,-0.000601
2016-04-06,0.010418,0.010637,0.010212,0.001397,0.007681,0.006047,0.003908,0.007191,0.005811,0.000853,...,0.023146,-0.000266,0.003543,0.004467,0.014389,0.005627,0.005413,0.013319,0.010458,0.007786
2016-04-07,-0.022051,-0.007282,-0.012046,-0.010389,-0.025662,-0.032162,-0.023679,-0.006518,-0.011949,-0.007329,...,-0.000527,-0.014057,-0.002098,-0.000662,-0.058840,-0.016660,-0.019326,-0.003446,-0.013175,-0.006824
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-24,0.000596,-0.033392,-0.027154,-0.002508,0.008552,0.012963,0.006780,0.007853,0.010957,0.008577,...,0.007670,0.021054,0.024288,-0.009915,0.013185,0.001136,-0.048113,-0.064327,-0.002331,-0.005781
2026-03-25,0.003887,0.001348,-0.004571,0.019668,0.010275,0.012592,-0.014462,-0.000711,0.008241,0.003875,...,-0.007962,0.003357,-0.008665,0.022733,0.070041,0.068418,-0.007301,-0.005809,-0.007398,0.005295
2026-03-26,0.001068,-0.031037,-0.013759,-0.042530,-0.012809,-0.010517,-0.006016,-0.008884,-0.007177,0.005399,...,0.012787,-0.022290,0.005521,0.005002,-0.077862,-0.067510,-0.022229,0.020022,0.003398,-0.010477


In [5]:
trading_days = 252
risk_free_rate = 0.03
mean_returns = returns_df.mean() * trading_days
cov_matrix = returns_df.cov() * trading_days
number_of_stocks = len(tickers)
weights = np.array([1 / number_of_stocks] * number_of_stocks)
returns = np.dot(weights, mean_returns)
std_dev = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
print(returns, std_dev)

0.13746587487538103 0.17513699215396591


In [6]:
def negative_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate=0.03):
    p_ret = np.dot(weights, mean_returns)
    p_std = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return -(p_ret - risk_free_rate) / p_std

args = (mean_returns, cov_matrix)
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
bounds = tuple((0, 1) for asset in range(number_of_stocks))
initial_guess = number_of_stocks * [1. / number_of_stocks]

In [7]:
optimal_portfolio = sco.minimize(
    negative_sharpe_ratio, 
    initial_guess, 
    args=args,
    method='SLSQP', 
    bounds=bounds, 
    constraints=constraints
)

optimal_weights = optimal_portfolio.x
optimal_return = np.sum(mean_returns * optimal_weights)
optimal_variance = np.dot(optimal_weights.T, np.dot(cov_matrix, optimal_weights))
optimal_volatility = np.sqrt(optimal_variance)
optimal_sharpe_ratio = optimal_return / optimal_volatility

print(f"Expected Return: {optimal_return:.4f}")
print(f"Variance: {optimal_variance:.4f}")

Expected Return: 0.2874
Variance: 0.0482


In [10]:
optimal_portfolio = pd.DataFrame(list(zip(tickers, optimal_weights)))
optimal_portfolio.to_csv('data/optimal_portfolio.csv', index=False)